In [ ]:
# Standard library imports
import sys
from pathlib import Path
from datetime import datetime
from time import sleep

# Add parent directory to path to import from src/
parent_dir = Path().resolve().parent
sys.path.insert(0, str(parent_dir))

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from xgboost import XGBRegressor
import joblib

# Local imports
from src.cpu_temp_bundled import HardwareMonitor

# Background color white
px.defaults.template = "plotly_white"

In [ ]:
# Standard library imports
from collections import deque
from datetime import datetime
from time import sleep

# Third-party imports
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from xgboost import XGBRegressor
import plotly.express as px
import joblib

# Local imports
from src.cpu_temp_bundled import HardwareMonitor


class CoreTempRegressor:
    """
    CPU Temperature anomaly detection using regression models.
    Can be used for training, saving/loading models, and real-time anomaly detection.
    """
    
    PLOT_STYLE = {
        'line_width': 3,
        'marker_size': 8,
        'threshold_line_width': 5,
        'threshold_opacity': 0.5,
        'error_color': 'orange',
        'lower_threshold_color': 'green',
        'upper_threshold_color': 'red',
        'line_dash': 'dash'
    }

    def __init__(self):
        self.scaler = None
        self.model = None
        self.data = None
        self.predict_data = None

        # Store for predictions
        self.x_test = None
        self.y_test = None
        self.y_pred = None
        self.test_data = None

        # Store feature engineering parameters
        self.lag_steps = None
        self.rolling_windows = None
        self.use_time_features = None

        # Store model name
        self.model_name = None
        self.multi_variable = True

        # Anomaly detection thresholds
        self.low_threshold = None
        self.high_threshold = None
        self.threshold_std = 1.5

        # Real-time monitoring buffer
        self.data_buffer = None
        self.time_counter = 0

        # Feature columns (set after training)
        self.feature_columns = None

    def configure_model(self, model='linear', multi_variable = True, scaler='standard', use_time_features=True, lag_steps=[1, 2, 3], rolling_windows=[3, 5, 7]):
        """Configure the model and scaler to use."""
        scaler_dict = {
            'standard': StandardScaler(),
            'minmax': MinMaxScaler(),
            'robust': RobustScaler()
        }

        model_dict = {
            'xgb': XGBRegressor(
                n_estimators=100,
                max_depth=3,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=3,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=42
            ),
            'linear': Ridge(
                alpha=1.0,
                solver='auto',
                random_state=42
            ),
            'lightgbm': LGBMRegressor(
                n_estimators=100,
                max_depth=3,
                learning_rate=0.1,
                num_leaves=7,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_samples=20,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=42,
                verbose=-1
            )
        }

        self.model = model_dict.get(model)
        self.scaler = scaler_dict.get(scaler)
        self.model_name = model
        self.lag_steps = lag_steps
        self.rolling_windows = rolling_windows
        self.use_time_features = use_time_features
        self.multi_variable = multi_variable

    def extract_CPU_data(self, iterations=100, interval=5, csv=False, progress_callback=None):
        """
        Extract CPU data for training.

        Args:
            iterations: Number of data points to collect
            interval: Time between collections (seconds)
            csv: If True, load from ../data/data.csv
            progress_callback: Optional callback function(current, total) for progress updates
        """
        if csv:
            self.data = pd.read_csv('../data/data.csv')
            if 'timestamp' in self.data.columns:
                self.data['timestamp'] = pd.to_datetime(self.data['timestamp'])
        else:
            data_list = []
            with HardwareMonitor() as monitor:
                for t in range(iterations):
                    row = monitor.get_essential_fast()
                    row['timestamp'] = datetime.now()
                    row['time'] = t
                    data_list.append(row)

                    if progress_callback:
                        progress_callback(t + 1, iterations)

                    if interval > 0:
                        sleep(interval)

            self.data = pd.DataFrame(data_list)
        return self.data
    
    def plot_CPU_data(self):
        px.line(self.data, x='timestamp', y=['cpu_temp','cpu_load','cpu_power','cpu_clock','cpu_volt','gpu_temp','gpu_load','gpu_power','mb_temp','ram_load','fan_rpm'], title='CPU Temperature').show()

    def create_time_features_on_df(self, df, lag_steps, rolling_windows):
        """Create lag, rolling, and diff features for better predictions."""
        exclude_cols = ['cpu_temp', 'time', 'fan_rpm', 'timestamp']
        feature_cols = [col for col in df.columns if col not in exclude_cols and df[col].dtype in ['float64', 'int64', 'float32', 'int32']]

        new_features = {}

        for col in feature_cols:
            # Lag features
            for lag in lag_steps:
                new_features[f'{col}_lag_{lag}'] = df[col].shift(lag)

            # Rolling statistics
            for window in rolling_windows:
                new_features[f'{col}_rolling_mean_{window}'] = df[col].rolling(window=window).mean()
                new_features[f'{col}_rolling_std_{window}'] = df[col].rolling(window=window).std()

            # Rate of change
            new_features[f'{col}_diff_1'] = df[col].diff(1)

        if new_features:
            new_cols_df = pd.DataFrame(new_features, index=df.index)
            df = pd.concat([df, new_cols_df], axis=1)

        return df

    def fit_predict(self, train_size=0.8, threshold_std=1.5):
        """
        Train the model and make predictions on test set.
        Also calculates anomaly thresholds.
        """
        self.threshold_std = threshold_std

        # Split the raw data FIRST (before feature engineering)
        split_idx = int(len(self.data) * train_size)
        train_data = self.data[:split_idx].copy()
        test_data = self.data[split_idx:].copy()

        # Apply time features SEPARATELY to train and test
        if self.use_time_features == True and self.multi_variable == True:
            train_data = self.create_time_features_on_df(train_data, self.lag_steps, self.rolling_windows)
            test_data = self.create_time_features_on_df(test_data, self.lag_steps, self.rolling_windows)

            train_data = train_data.dropna()
            test_data = test_data.dropna()

        # Prepare X and y
        if self.multi_variable == True:
            x_train = train_data.drop(['cpu_temp', 'time', 'fan_rpm', 'timestamp'], axis=1)
            y_train = train_data['cpu_temp']

            self.x_test = test_data.drop(['cpu_temp', 'time', 'fan_rpm', 'timestamp'], axis=1)
            self.y_test = test_data['cpu_temp']
        
        else:
            x_train = train_data[['time']]
            y_train = train_data['cpu_temp']

            self.x_test = test_data[['time']]
            self.y_test = test_data['cpu_temp']

        # Store feature columns for later use
        self.feature_columns = list(x_train.columns)

        # Fit scaler ONLY on training data
        x_train_scaled = self.scaler.fit_transform(x_train)
        x_test_scaled = self.scaler.transform(self.x_test)

        # Train model
        self.model.fit(x_train_scaled, y_train)

        # Make predictions
        self.y_pred = self.model.predict(x_test_scaled)

        # Store test_data for plotting
        self.test_data = test_data

        # Calculate anomaly thresholds
        diff = self.y_test.values - self.y_pred
        mean_diff = np.mean(diff)
        std_diff = np.std(diff)
        self.low_threshold = mean_diff - threshold_std * std_diff
        self.high_threshold = mean_diff + threshold_std * std_diff

    def predict(self, X):
        """Make predictions on new data."""
        X_scaled = self.scaler.transform(X)
        self.predict_data = self.model.predict(X_scaled)
        return self.predict_data

    def get_thresholds(self):
        """Return current anomaly thresholds."""
        return {
            'low': self.low_threshold,
            'high': self.high_threshold,
            'std_multiplier': self.threshold_std
        }

    def init_realtime_buffer(self):
        """Initialize the buffer for real-time anomaly detection."""
        max_window = max(self.rolling_windows) if self.rolling_windows else 7
        max_lag = max(self.lag_steps) if self.lag_steps else 3
        buffer_size = max(max_window, max_lag) + 5  # Extra buffer
        self.data_buffer = deque(maxlen=buffer_size)
        self.time_counter = 0

    def detect_anomaly(self, current_data: dict) -> tuple:
        """
        Detect if current data point is an anomaly.

        Args:
            current_data: Dict with sensor readings from HardwareMonitor.get_essential_fast()

        Returns:
            tuple: (is_anomaly: bool, diff: float, predicted: float, actual: float)
        """
        if self.data_buffer is None:
            self.init_realtime_buffer()

        # Add timestamp and time
        current_data = current_data.copy()
        current_data['timestamp'] = datetime.now()
        current_data['time'] = self.time_counter
        self.time_counter += 1

        # Add to buffer
        self.data_buffer.append(current_data)

        # Need enough history for features
        min_required = max(max(self.rolling_windows), max(self.lag_steps)) + 1
        if len(self.data_buffer) < min_required:
            return False, 0.0, 0.0, current_data.get('cpu_temp', 0.0)

        # Create DataFrame from buffer
        df = pd.DataFrame(list(self.data_buffer))

        # Apply time features
        if self.use_time_features == True and self.multi_variable == True:
            df = self.create_time_features_on_df(df, self.lag_steps, self.rolling_windows)
            df = df.dropna()

        if len(df) == 0:
            return False, 0.0, 0.0, current_data.get('cpu_temp', 0.0)

        # Get last row for prediction
        last_row = df.iloc[[-1]]

        # Prepare features
        if self.multi_variable == True:
            X = last_row.drop(['cpu_temp', 'time', 'fan_rpm', 'timestamp'], axis=1, errors='ignore')

        else:
            X = last_row[['time']]
            
        # Ensure columns match training
        if self.feature_columns:
            missing_cols = set(self.feature_columns) - set(X.columns)
            for col in missing_cols:
                X[col] = 0
            X = X[self.feature_columns]

        # Predict
        predicted = self.predict(X)[0]
        actual = current_data.get('cpu_temp', 0.0)
        diff = actual - predicted

        # Check anomaly
        is_anomaly = diff < self.low_threshold or diff > self.high_threshold

        return is_anomaly, diff, predicted, actual
    
    def plot_predictions(self, threshold_std=1.5):
        # Use stored test_data time values
        time_test = self.test_data['time'].values
        timestamp_test = self.test_data['timestamp'].values

        # Get model name for titles
        model_display_name = self.model_name.upper() if self.model_name else "Model"

        # Prepare data
        df_test = self.x_test.copy()
        df_test['time'] = time_test
        df_test['timestamp'] = timestamp_test
        df_test["Actual"] = self.y_test.values
        df_test["Predicted"] = self.y_pred
        df_test = df_test.sort_values("time")

        df_test["diff"] = df_test["Actual"] - df_test["Predicted"]
        mean_diff = df_test["diff"].mean()
        stardard_dev = df_test["diff"].std()
        low_lim = mean_diff - threshold_std * stardard_dev
        high_lim = mean_diff + threshold_std * stardard_dev

        # Transform to long format (better for px.line)
        df_plot = df_test.melt(id_vars="timestamp", value_vars=["diff"], var_name="Type", value_name="Temperature Diff")

        # Predictions plot
        fig_pred = px.line(df_test, x="timestamp", y=["Actual", "Predicted"], title=f"{model_display_name} Regression: CPU Temp: Predicted vs Actual")
        fig_pred.update_traces(
            line_width=self.PLOT_STYLE['line_width'], 
            marker=dict(size=self.PLOT_STYLE['marker_size'])
        )
        fig_pred.show()

        # Error plot
        fig = px.line(df_plot, x="timestamp", y="Temperature Diff", color="Type", title=f"{model_display_name} Regression: CPU Temp Difference: Predicted vs Actual")
        fig.update_traces(
            line_color=self.PLOT_STYLE['error_color'], 
            line_width=self.PLOT_STYLE['line_width'], 
            marker=dict(size=self.PLOT_STYLE['marker_size'])
        )
        fig.add_hline(
            y=low_lim, 
            line_width=self.PLOT_STYLE['threshold_line_width'], 
            line_color=self.PLOT_STYLE['lower_threshold_color'], 
            line_dash=self.PLOT_STYLE['line_dash'], 
            opacity=self.PLOT_STYLE['threshold_opacity']
        )
        fig.add_hline(
            y=high_lim, 
            line_width=self.PLOT_STYLE['threshold_line_width'], 
            line_color=self.PLOT_STYLE['upper_threshold_color'], 
            line_dash=self.PLOT_STYLE['line_dash'], 
            opacity=self.PLOT_STYLE['threshold_opacity']
        )
        fig.show()

        # Anomaly detection plot
        df_test["anomaly"] = (df_test["diff"] > high_lim) | (df_test["diff"] < low_lim)
        fig_anom = px.line(df_test, x="timestamp", y="anomaly", color="anomaly", title=f"{model_display_name} Regression: Anomaly Detection", markers=True)
        fig_anom.show()

    def evaluate_metrics(self):
        """Calculate and return evaluation metrics."""
        rmse = np.sqrt(mean_squared_error(self.y_test, self.y_pred))
        mape = np.mean(np.abs((self.y_test - self.y_pred) / self.y_test)) * 100
        r2 = r2_score(self.y_test, self.y_pred)
        mae = mean_absolute_error(self.y_test, self.y_pred)

        return {
            'rmse': rmse,
            'mape': mape,
            'r2': r2,
            'mae': mae
        }

    def save_model(self, path: str):
        """Save the trained model to file."""
        joblib.dump(self, path)

    @staticmethod
    def load_model(path: str) -> 'CoreTempRegressor':
        """Load a trained model from file."""
        model = joblib.load(path)
        # Initialize buffer for real-time detection
        model.init_realtime_buffer()
        return model

In [ ]:
# Instanciating the function
core_predictor = CoreTempRegressor()
core_predictor.extract_CPU_data(iterations = 50000, interval = 0, csv = True)
core_predictor.plot_CPU_data()

## Example Usage with Time-Based Features

The new workflow includes feature engineering for better predictions:

In [ ]:
# Training Linear model
core_predictor.configure_model(model="linear", scaler="standard", multi_variable = False, use_time_features=False, lag_steps=[1, 2, 3], rolling_windows=[3, 5, 7])
core_predictor.fit_predict(train_size=0.8)
core_predictor.plot_predictions()
core_predictor.evaluate_metrics()

# Save the trained model
joblib.dump(core_predictor, '../models/cpu_temp_model_linear.joblib')

In [ ]:
# Training XGBoost model
core_predictor.configure_model(model="xgb", scaler="standard", multi_variable = False, use_time_features=False, lag_steps=[1, 2, 3], rolling_windows=[3, 5, 7])
core_predictor.fit_predict(train_size=0.6)
core_predictor.plot_predictions()
core_predictor.evaluate_metrics()

# Save the trained model
joblib.dump(core_predictor, '../models/cpu_temp_model_xgb.joblib')

In [ ]:
# Training LightGBM model
core_predictor.configure_model(model="lightgbm", scaler="standard", multi_variable = False, use_time_features=False, lag_steps=[1, 2, 3], rolling_windows=[3, 5, 7])
core_predictor.fit_predict(train_size=0.8)
core_predictor.plot_predictions()
core_predictor.evaluate_metrics()

# Save the trained model
joblib.dump(core_predictor, '../models/cpu_temp_model_lightgbm.joblib')